<a href="https://colab.research.google.com/github/asif851/Clinical-Text-Classification/blob/main/Clinical_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
df = pd.read_csv('mtsamples.csv', index_col=0)

In [22]:
df.shape
df.head()
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4999 entries, 0 to 4998
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   description        4999 non-null   object
 1   medical_specialty  4999 non-null   object
 2   sample_name        4999 non-null   object
 3   transcription      4966 non-null   object
 4   keywords           3931 non-null   object
dtypes: object(5)
memory usage: 234.3+ KB


In [23]:
counts=df['medical_specialty'].value_counts()
counts

,count
medical_specialty,
Surgery,1103
Consult - History and Phy.,516
Cardiovascular / Pulmonary,372
Orthopedic,355
Radiology,273
General Medicine,259
Gastroenterology,230
Neurology,223
SOAP / Chart / Progress Notes,166


In [24]:
df['medical_specialty'].nunique()

40

In [25]:
base = counts.iloc[0]

ratio = counts / base
print(ratio)

medical_specialty
Surgery                          1.000000
Consult - History and Phy.       0.467815
Cardiovascular / Pulmonary       0.337262
Orthopedic                       0.321850
Radiology                        0.247507
General Medicine                 0.234814
Gastroenterology                 0.208522
Neurology                        0.202176
SOAP / Chart / Progress Notes    0.150499
Obstetrics / Gynecology          0.145059
Urology                          0.143246
Discharge Summary                0.097915
ENT - Otolaryngology             0.088849
Neurosurgery                     0.085222
Hematology - Oncology            0.081596
Ophthalmology                    0.075249
Nephrology                       0.073436
Emergency Room Reports           0.067996
Pediatrics - Neonatal            0.063463
Pain Management                  0.056210
Psychiatry / Psychology          0.048051
Office Notes                     0.046238
Podiatry                         0.042611
Dermatology     

In [26]:
df['medical_specialty'] = df['medical_specialty'].str.strip()
top_classes = df['medical_specialty'].value_counts().index[:15]
dff = df[df['medical_specialty'].isin(top_classes)]
dff_counts=dff['medical_specialty'].value_counts()
dff_counts

,count
medical_specialty,
Surgery,1103
Consult - History and Phy.,516
Cardiovascular / Pulmonary,372
Orthopedic,355
Radiology,273
General Medicine,259
Gastroenterology,230
Neurology,223
SOAP / Chart / Progress Notes,166


In [27]:
dff.shape

(4205, 5)

In [28]:
dff=dff.dropna(subset=['transcription'])
dff.shape

(4174, 5)

In [29]:
dff['sample_name'].head(10)

,sample_name
3,2-D Echocardiogram - 1
4,2-D Echocardiogram - 2
7,2-D Echocardiogram - 3
9,2-D Echocardiogram - 4
11,2-D Doppler
12,Moyamoya Disease
16,Tracheostomy
18,Vasectomy - 4
19,Airway Compromise & Foreign Body - ER Visit
20,Whole Body Radionuclide Bone Scan


In [30]:
dff = dff[['transcription', 'medical_specialty']]
dff.shape

(4174, 2)

In [31]:
print(dff['transcription'].iloc[0])

2-D M-MODE: , ,1.  Left atrial enlargement with left atrial diameter of 4.7 cm.,2.  Normal size right and left ventricle.,3.  Normal LV systolic function with left ventricular ejection fraction of 51%.,4.  Normal LV diastolic function.,5.  No pericardial effusion.,6.  Normal morphology of aortic valve, mitral valve, tricuspid valve, and pulmonary valve.,7.  PA systolic pressure is 36 mmHg.,DOPPLER: , ,1.  Mild mitral and tricuspid regurgitation.,2.  Trace aortic and pulmonary regurgitation.


In [32]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [33]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import re

stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()

    text = re.sub(r'[^a-z0-9\s]', '', text)


    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    text = ' '.join(words)

    return text

In [34]:
print(clean_text(dff['transcription'].iloc[0]))

2d mmode 1 left atrial enlargement left atrial diameter 47 cm2 normal size right left ventricle3 normal lv systolic function left ventricular ejection fraction 514 normal lv diastolic function5 pericardial effusion6 normal morphology aortic valve mitral valve tricuspid valve pulmonary valve7 pa systolic pressure 36 mmhgdoppler 1 mild mitral tricuspid regurgitation2 trace aortic pulmonary regurgitation


In [35]:
dff['clean_transcription']=dff['transcription'].apply(clean_text)
dff.head(5)

,transcription,medical_specialty,clean_transcription
3,"2-D M-MODE: , ,1. Left atrial enlargement wit...",Cardiovascular / Pulmonary,2d mmode 1 left atrial enlargement left atrial...
4,1. The left ventricular cavity size and wall ...,Cardiovascular / Pulmonary,1 left ventricular cavity size wall thickness ...
7,"2-D ECHOCARDIOGRAM,Multiple views of the heart...",Cardiovascular / Pulmonary,2d echocardiogrammultiple view heart great ves...
9,"DESCRIPTION:,1. Normal cardiac chambers size....",Cardiovascular / Pulmonary,description1 normal cardiac chamber size2 norm...
11,"2-D STUDY,1. Mild aortic stenosis, widely calc...",Cardiovascular / Pulmonary,2d study1 mild aortic stenosis widely calcifie...


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2), min_df=5)
X = vectorizer.fit_transform(dff['clean_transcription'])
dff['medical_specialty'] = dff['medical_specialty'].str.strip()
y=dff['medical_specialty']

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(3339, 1000) (3339,)
(835, 1000) (835,)


In [38]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1, penalty='l2', solver='liblinear')
model.fit(X_train, y_train)

LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000,
                   solver='liblinear')

In [39]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
param_grid = [
    {'solver': ['lbfgs', 'saga'], 'penalty': ['l2'], 'C': [0.001, 0.01, 0.1, 1.0]},
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.001, 0.01, 0.1, 1.0]},
]

lr_base = LogisticRegression(max_iter=1000, class_weight='balanced')
grid_search = GridSearchCV(lr_base, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Best cross-validation score: 0.4483387048990474


In [40]:
best_model = grid_search.best_estimator_
test_acc = accuracy_score(y_test, best_model.predict(X_test))
train_acc = accuracy_score(y_train, best_model.predict(X_train))
print("Train:", train_acc)
print("Test:", test_acc)
print("Gap:", train_acc - test_acc)

Train: 0.4983528002395927
Test: 0.444311377245509
Gap: 0.05404142299408371


In [41]:
from sklearn.metrics import accuracy_score, classification_report
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.444311377245509
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.47      0.46      0.47        74
   Consult - History and Phy.       0.45      0.56      0.50       103
            Discharge Summary       0.41      0.90      0.57        21
         ENT - Otolaryngology       0.52      0.68      0.59        19
             Gastroenterology       0.44      0.36      0.40        45
             General Medicine       0.33      0.25      0.29        52
        Hematology - Oncology       0.55      0.33      0.41        18
                    Neurology       0.46      0.56      0.51        45
                 Neurosurgery       0.24      0.47      0.32        19
      Obstetrics / Gynecology       0.55      0.68      0.61        31
                   Orthopedic       0.33      0.25      0.29        71
                    Radiology       0.38      0.38      0.38        55
SOAP / Chart / Progress Notes       0.36      0.39      0.

In [42]:
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)
print("Train accuracy:", accuracy_score(y_train, train_preds))
print("Test accuracy:", accuracy_score(y_test, test_preds))

Train accuracy: 0.4983528002395927
Test accuracy: 0.444311377245509


In [43]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted')
print(scores)
print(scores.mean())

[0.44529139 0.42683264 0.45966031 0.4313002  0.44280282]
0.4411774713672695


In [44]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [45]:
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.4287425149700599
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.45      0.39      0.42        74
   Consult - History and Phy.       0.36      0.92      0.51       103
            Discharge Summary       0.43      0.14      0.21        21
         ENT - Otolaryngology       1.00      0.05      0.10        19
             Gastroenterology       0.38      0.22      0.28        45
             General Medicine       0.00      0.00      0.00        52
        Hematology - Oncology       0.00      0.00      0.00        18
                    Neurology       0.46      0.47      0.46        45
                 Neurosurgery       0.00      0.00      0.00        19
      Obstetrics / Gynecology       0.56      0.16      0.25        31
                   Orthopedic       0.37      0.31      0.34        71
                    Radiology       0.38      0.33      0.35        55
SOAP / Chart / Progress Notes       0.00      0.00      0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [46]:
from sklearn.svm import LinearSVC
model = LinearSVC(class_weight='balanced', C=0.01, loss='squared_hinge', max_iter=500)
model.fit(X_train, y_train)

LinearSVC(C=0.01, class_weight='balanced', max_iter=500)

In [47]:
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.4407185628742515
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.44      0.49      0.46        74
   Consult - History and Phy.       0.49      0.50      0.50       103
            Discharge Summary       0.40      0.90      0.56        21
         ENT - Otolaryngology       0.52      0.84      0.64        19
             Gastroenterology       0.44      0.51      0.47        45
             General Medicine       0.35      0.27      0.30        52
        Hematology - Oncology       0.42      0.44      0.43        18
                    Neurology       0.43      0.58      0.49        45
                 Neurosurgery       0.26      0.53      0.34        19
      Obstetrics / Gynecology       0.52      0.71      0.60        31
                   Orthopedic       0.39      0.37      0.38        71
                    Radiology       0.39      0.36      0.38        55
SOAP / Chart / Progress Notes       0.32      0.36      0

In [48]:
for c in [0.01, 0.05, 0.1, 0.5, 1.0]:
    svm = LinearSVC(class_weight='balanced', C=c)
    svm.fit(X_train, y_train)
    preds = svm.predict(X_test)
    print(f"C={c}: {accuracy_score(y_test, preds):.4f}")

C=0.01: 0.4407
C=0.05: 0.3952
C=0.1: 0.3760
C=0.5: 0.2994
C=1.0: 0.2527


In [49]:
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)
print("Train accuracy:", accuracy_score(y_train, train_preds))
print("Test accuracy:", accuracy_score(y_test, test_preds))

Train accuracy: 0.4980533093740641
Test accuracy: 0.4407185628742515


In [50]:
for c in [0.01, 0.05, 0.1, 0.5, 1.0]:
    model=LogisticRegression(max_iter=1000, class_weight='balanced', C=c)
    model.fit(X_train, y_train)
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    print(f"C={c} -> Train: {accuracy_score(y_train, train_preds):.4f}, Test: {accuracy_score(y_test, test_preds):.4f}, Gap: {accuracy_score(y_train, train_preds) - accuracy_score(y_test, test_preds):.4f}")

C=0.01 -> Train: 0.4471, Test: 0.4108, Gap: 0.0364
C=0.05 -> Train: 0.4567, Test: 0.4192, Gap: 0.0376
C=0.1 -> Train: 0.4657, Test: 0.4204, Gap: 0.0453
C=0.5 -> Train: 0.5013, Test: 0.4036, Gap: 0.0978
C=1.0 -> Train: 0.5214, Test: 0.3856, Gap: 0.1358


In [51]:
from sklearn.linear_model import PassiveAggressiveClassifier
model = PassiveAggressiveClassifier(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.16407185628742516
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.18      0.18      0.18        74
   Consult - History and Phy.       0.19      0.16      0.17       103
            Discharge Summary       0.30      0.33      0.32        21
         ENT - Otolaryngology       0.32      0.37      0.34        19
             Gastroenterology       0.11      0.11      0.11        45
             General Medicine       0.10      0.12      0.11        52
        Hematology - Oncology       0.10      0.17      0.12        18
                    Neurology       0.15      0.16      0.15        45
                 Neurosurgery       0.07      0.16      0.10        19
      Obstetrics / Gynecology       0.24      0.29      0.26        31
                   Orthopedic       0.14      0.13      0.13        71
                    Radiology       0.15      0.20      0.17        55
SOAP / Chart / Progress Notes       0.12      0.12      

In [52]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.17964071856287425
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.15      0.15      0.15        74
   Consult - History and Phy.       0.28      0.32      0.30       103
            Discharge Summary       0.20      0.19      0.20        21
         ENT - Otolaryngology       0.17      0.11      0.13        19
             Gastroenterology       0.05      0.04      0.05        45
             General Medicine       0.10      0.08      0.09        52
        Hematology - Oncology       0.08      0.11      0.09        18
                    Neurology       0.11      0.11      0.11        45
                 Neurosurgery       0.00      0.00      0.00        19
      Obstetrics / Gynecology       0.12      0.10      0.11        31
                   Orthopedic       0.05      0.04      0.05        71
                    Radiology       0.11      0.13      0.12        55
SOAP / Chart / Progress Notes       0.22      0.24      

In [53]:
#label encoder for XG_Boost
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [54]:
from xgboost import XGBClassifier
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train, y_train_encoded)
predictions = model.predict(X_test)
print(accuracy_score(y_test_encoded, predictions))
print(classification_report(y_test_encoded, predictions))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:09:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.17724550898203592
              precision    recall  f1-score   support

           0       0.14      0.14      0.14        74
           1       0.25      0.29      0.27       103
           2       0.24      0.24      0.24        21
           3       0.20      0.11      0.14        19
           4       0.07      0.07      0.07        45
           5       0.11      0.10      0.10        52
           6       0.05      0.06      0.05        18
           7       0.15      0.16      0.15        45
           8       0.00      0.00      0.00        19
           9       0.05      0.03      0.04        31
          10       0.03      0.03      0.03        71
          11       0.11      0.13      0.12        55
          12       0.18      0.18      0.18        33
          13       0.28      0.32      0.30       218
          14       0.00      0.00      0.00        31

    accuracy                           0.18       835
   macro avg       0.12      0.12      0.12       835
weight

In [55]:
from sklearn.metrics import accuracy_score
import numpy as np

def confidence_interval(acc, n):
    margin = 1.96 * np.sqrt((acc * (1 - acc)) / n)
    return acc - margin, acc + margin

n = len(y_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1, penalty='l2', solver='liblinear'),
    'LinearSVC': LinearSVC(class_weight='balanced', C=0.01, loss='squared_hinge', max_iter=500),
    'Naive Bayes': MultinomialNB(),
    'Passive Aggressive': PassiveAggressiveClassifier(class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(class_weight='balanced')
}

print(f"{'Model':30s} | {'Train':6s} | {'Test':6s} | {'Gap':6s} | {'95% CI':20s}")
print("-" * 85)

for name, m in models.items():
    m.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, m.predict(X_train))
    test_acc = accuracy_score(y_test, m.predict(X_test))
    gap = train_acc - test_acc
    lower, upper = confidence_interval(test_acc, n)
    print(f"{name:30s} | {train_acc:.4f} | {test_acc:.4f} | {gap:.4f} | ({lower:.4f}, {upper:.4f})")

Model                          | Train  | Test   | Gap    | 95% CI              
-------------------------------------------------------------------------------------
Logistic Regression            | 0.4984 | 0.4443 | 0.0540 | (0.4106, 0.4780)
LinearSVC                      | 0.4981 | 0.4407 | 0.0573 | (0.4070, 0.4744)
Naive Bayes                    | 0.4663 | 0.4287 | 0.0376 | (0.3952, 0.4623)
Passive Aggressive             | 0.6077 | 0.1856 | 0.4220 | (0.1593, 0.2120)
Random Forest                  | 0.6178 | 0.1832 | 0.4346 | (0.1570, 0.2095)
